# 🌱 Crop Disease & Pest Classifier — Colab / Kaggle Training Pipeline

This notebook trains and exports the **Crop Disease & Pest Classifier** on a free Google Colab or Kaggle GPU.

### Pipeline Overview
1. **Environment Setup & GPU Check** (PyTorch 2.x + timm + Albumentations)
2. **Download & Ingest Public Datasets** (PlantVillage, PlantDoc, IP102)
3. **Clean & Leak-Free Perceptual Splitting** (pHash Hamming distance $\le 6$)
4. **Two-Stage Supervised Training** (Frozen head $\to$ Discriminative LR fine-tuning with AMP, EMA, AdamW)
5. **Calibration, OOD & Grad-CAM Evaluation**
6. **Export Dual-Output ONNX & CPU Benchmark**

In [ ]:
# Step 1: Check GPU environment
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 2: Install dependencies
!pip install -q timm albumentations opencv-python-headless Pillow ImageHash pandas pyarrow scikit-learn scipy onnx onnxruntime optuna PyYAML tqdm

In [ ]:
# Step 3: Download PlantVillage & PlantDoc
!python -m ml.data.download --datasets plantvillage plantdoc ood_non_plant

In [ ]:
# Step 4: Clean, deduplicate and split with 0 leak pHash clustering
!python -m ml.data.clean_and_merge
!python -m ml.data.split
!python -m ml.data.dataset_card

In [ ]:
# Step 5: Two-stage training with EfficientNetV2-S (or ConvNeXt-Tiny)
!python -m ml.training.train --config configs/train_efficientnetv2_s.yaml

In [ ]:
# Step 6: Evaluate In-Domain vs. PlantDoc Real-Field Gap & Temperature Calibration
!python -m ml.evaluation.evaluate --checkpoint artifacts/best_model.pt

In [ ]:
# Step 7: Export to dual-output ONNX and benchmark CPU latency
!python -m ml.export.export_model --checkpoint artifacts/best_model.pt --torchscript --quantize-int8
!python -m ml.export.benchmark_cpu --onnx artifacts/model.onnx --checkpoint artifacts/best_model.pt

In [ ]:
# Step 8: Download production bundle to local machine
from google.colab import files
files.download('artifacts/model.onnx')
files.download('artifacts/model_meta.json')